# XLZD CNP Workflow

This is the main CNP notebook. Use `EXPERIMENT` in the setup cell to switch between:

- `default`
- `minibatch`
- `fixedcontext`
- `fullpass`

This notebook assumes the HDF5 files already exist in:

- `outputs/training/lf/*.h5`
- `outputs/training/hf/*.h5`
- `outputs/validation/hf/*.h5`

It runs only the CNP part of the RESuM workflow:

1. load the XLZD CNP settings
2. train the CNP on training LF files
3. predict on training LF+HF files
4. predict on validation HF files
5. inspect the saved CSVs and plots

Older fixed-path notebooks are kept under `src/run_cnp/additional_experiments/` for reference only.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_cnp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_cnp"))

from cnp_clean_pipeline import load_runtime_config, predict_cnp, train_cnp

EXPERIMENT = "minibatch"  # one of: "default", "minibatch", "fixedcontext", "fullpass"

CONFIG_BY_EXPERIMENT = {
    "default": REPO_ROOT / "src" / "xlzd" / "settings.yaml",
    "minibatch": REPO_ROOT / "src" / "xlzd" / "settings_minibatch.yaml",
    "fixedcontext": REPO_ROOT / "src" / "xlzd" / "settings_fixedcontext.yaml",
    "fullpass": REPO_ROOT / "src" / "xlzd" / "settings_fullpass.yaml",
}
VALIDATION_CONFIG_BY_EXPERIMENT = {
    "default": REPO_ROOT / "src" / "xlzd" / "settings_validation.yaml",
    "minibatch": REPO_ROOT / "src" / "xlzd" / "settings_validation_minibatch.yaml",
    "fixedcontext": REPO_ROOT / "src" / "xlzd" / "settings_validation_fixedcontext.yaml",
    "fullpass": REPO_ROOT / "src" / "xlzd" / "settings_validation_fullpass.yaml",
}

if EXPERIMENT not in CONFIG_BY_EXPERIMENT:
    raise ValueError(f"Unknown EXPERIMENT={EXPERIMENT!r}. Choose from {sorted(CONFIG_BY_EXPERIMENT)}")

CONFIG_PATH = CONFIG_BY_EXPERIMENT[EXPERIMENT]
VALIDATION_CONFIG_PATH = VALIDATION_CONFIG_BY_EXPERIMENT[EXPERIMENT]

SEED = 42
STEPS_PER_EPOCH = 5000
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
REPR_DIM = 32
HIDDEN = 128
DROPOUT = 0.1
MONITOR_EVERY_BY_EXPERIMENT = {"default": 5000, "minibatch": 5000, "fixedcontext": 5000, "fullpass": 500}
MONITOR_EVERY = MONITOR_EVERY_BY_EXPERIMENT[EXPERIMENT]
MC_SAMPLES = 30
CHUNK_SIZE = 20000

print(f"Repo root: {REPO_ROOT}")
print(f"Experiment: {EXPERIMENT}")
print(f"Training config: {CONFIG_PATH}")
print(f"Validation config: {VALIDATION_CONFIG_PATH}")


## 1. Load And Inspect The Runtime Config

This cell loads the CNP settings and shows the folders the notebook will use.


In [ ]:
runtime = load_runtime_config(CONFIG_PATH, seed=SEED)
validation_runtime = load_runtime_config(VALIDATION_CONFIG_PATH, seed=SEED)

summary = pd.DataFrame(
    {
        "field": [
            "version",
            "train_dir",
            "predict_dirs",
            "theta_headers",
            "phi_headers",
            "target_headers",
            "training_mode",
            "epochs",
            "steps_per_epoch",
            "context_ratio",
            "out_dir",
        ],
        "value": [
            runtime.version,
            str(runtime.train_dir),
            ", ".join(str(p) for p in runtime.predict_dirs),
            ", ".join(runtime.theta_headers),
            ", ".join(runtime.phi_headers),
            ", ".join(runtime.target_headers),
            runtime.training_mode,
            runtime.epochs,
            runtime.steps_per_epoch,
            runtime.context_ratio,
            str(runtime.out_dir),
        ],
    }
)
summary


## 2. Train The CNP

This trains the CNP on the LF training H5 files and writes the model, history CSV, and plots to `data/out/cnp`.

If you want a smaller first run, reduce `STEPS_PER_EPOCH` above before executing this cell.


In [ ]:
train_result = train_cnp(
    runtime,
    steps_per_epoch=STEPS_PER_EPOCH,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    repr_dim=REPR_DIM,
    hidden=HIDDEN,
    dropout=DROPOUT,
    monitor_every=MONITOR_EVERY,
    show_monitor_plots=True,
)

pd.DataFrame(
    {
        "artifact": ["model_path", "history_csv", "history_plot", "sample_plot"],
        "path": [
            str(train_result.model_path),
            str(train_result.history_csv),
            str(train_result.history_plot),
            str(train_result.sample_plot),
        ],
    }
)


## 3. Predict On Training LF + HF

This aggregates one prediction row per theta file and writes the output CSV and heatmaps to `data/out/cnp`.


In [ ]:
predict_result_train = predict_cnp(
    runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

train_pred_df = pd.read_csv(predict_result_train.csv_path)
display(train_pred_df.head())
display(train_pred_df.describe(include="all"))


In [ ]:
display(Image(filename=str(predict_result_train.heatmap_path)))
display(Image(filename=str(predict_result_train.error_heatmap_path)))


## 4. Predict On Held-Out HF Validation Files

This reuses the trained model and writes a validation CSV and validation heatmaps.


In [ ]:
predict_result_validation = predict_cnp(
    validation_runtime,
    model_path=train_result.model_path,
    mc_samples=MC_SAMPLES,
    chunk_size=CHUNK_SIZE,
)

validation_pred_df = pd.read_csv(predict_result_validation.csv_path)
display(validation_pred_df.head())
display(validation_pred_df.describe(include="all"))


In [ ]:
display(Image(filename=str(predict_result_validation.heatmap_path)))
display(Image(filename=str(predict_result_validation.error_heatmap_path)))


## 5. Key Output Files

The notebook writes the same artifacts as the script pipeline. The most important ones are:

- model checkpoint: `data/out/cnp/cnp_<version>_model_<epochs>epochs.pth`
- training history CSV: `data/out/cnp/cnp_<version>_history_<epochs>epochs.csv`
- training LF+HF aggregated prediction CSV: `data/out/cnp/cnp_<version>_output_<epochs>epochs.csv`
- validation HF aggregated prediction CSV: `data/out/cnp/cnp_<version>_output_validation_<epochs>epochs.csv`
- training and validation heatmaps saved alongside those CSVs
